## REPARTATION


In [0]:
df = spark.table("workspace.default.movies")
df.display()

title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
Pather Panchali,Bollywood,1955,8.3,Government of West Bengal,70000.0,100000.0,Thousands,INR,Bengali
Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.0,954.8,Millions,USD,English
Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.0,644.8,Millions,USD,English
Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.0,854.0,Millions,USD,English
Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.0,670.0,Millions,USD,English
The Shawshank Redemption,Hollywood,1994,9.3,Castle Rock Entertainment,25.0,73.3,Millions,USD,English
Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.0,701.8,Millions,USD,English
The Pursuit of Happyness,Hollywood,2006,8.0,Columbia Pictures,55.0,307.1,Millions,USD,English
Gladiator,Hollywood,2000,8.5,Universal Pictures,103.0,460.5,Millions,USD,English
Titanic,Hollywood,1997,7.9,Paramount Pictures,200.0,2202.0,Millions,USD,English


In [0]:
%sql
select distinct studio from workspace.default.movies

studio
Government of West Bengal
Marvel Studios
Castle Rock Entertainment
Warner Bros. Pictures
Columbia Pictures
Universal Pictures
Paramount Pictures
Liberty Films
20th Century Fox
Syncopy


In [0]:
rep_by_key = df.repartition(6,"studio")
rep_by_key.show()

+--------------------+---------+------------+-----------+--------------------+------+-------+--------+--------+--------+
|               title| industry|release_year|imdb_rating|              studio|budget|revenue|    unit|currency|language|
+--------------------+---------+------------+-----------+--------------------+------+-------+--------+--------+--------+
|Doctor Strange in...|Hollywood|        2022|        7.0|      Marvel Studios| 200.0|  954.8|Millions|     USD| English|
|Thor: The Dark Wo...|Hollywood|        2013|        6.8|      Marvel Studios| 165.0|  644.8|Millions|     USD| English|
|     Thor: Ragnarok |Hollywood|        2017|        7.9|      Marvel Studios| 180.0|  854.0|Millions|     USD| English|
|Thor: Love and Th...|Hollywood|        2022|        6.8|      Marvel Studios| 250.0|  670.0|Millions|     USD| English|
|        Interstellar|Hollywood|        2014|        8.6|Warner Bros. Pict...| 165.0|  701.8|Millions|     USD| English|
|     The Dark Knight|Hollywood|

In [0]:
rep_by_key.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonShuffleExchangeSource
         +- PhotonShuffleMapStage REPARTITION_BY_NUM, [id=#7310]
            +- PhotonShuffleExchangeSink hashpartitioning(studio#11217, 6)
               +- PhotonScan parquet workspace.default.movies[title#11213,industry#11214,release_year#11215L,imdb_rating#11216,studio#11217,budget#11218,revenue#11219,unit#11220,currency#11221,language#11222] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-wng4n/uc/ac94fad9-4917-42a3-8b8c-79891d1ea9e4..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<title:string,industry:string,release_year:bigint,imdb_rating:double,studio:string,budget:d..., RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  miss

In [0]:
out_path = "/Volumes/workspace/default/repartition_demo/repartition_6"
rep_by_key.write.mode("overwrite").parquet(out_path)

## coalesce